# 05 — Macro Velocity Dashboard

GDP, CPI, PMI, employment — which macro indicators are inflecting?
Requires free FRED key: https://fred.stlouisfed.org/docs/api/api_key.html

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'specvel'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
sns.set_theme(style='darkgrid')

FRED_KEY = os.environ.get('FRED_API_KEY', 'your_key_here')
if FRED_KEY == 'your_key_here':
    raise ValueError('Set FRED_KEY. Free at: https://fred.stlouisfed.org/docs/api/api_key.html')

from adapters.macro import MacroAdapter
from core import compute_velocity
from leaderboard import run_leaderboard, print_leaderboard, save_leaderboard

START = '2010-01-01'
END   = '2026-03-10'

In [ ]:
adapter = MacroAdapter(api_key=FRED_KEY)
df = run_leaderboard(adapter, start=START, end=END,
                     asset_class='macro', top_n=20, lookback=12)
print_leaderboard(df, title='MACRO VELOCITY LEADERBOARD')

## Inflation Velocity — CPI vs Core CPI vs PCE

In [ ]:
inflation_series = {
    'CPIAUCSL':  'CPI All Items',
    'CPILFESL':  'Core CPI',
    'PCEPILFE':  'Core PCE',
}

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

for ax, (sid, label) in zip(axes, inflation_series.items()):
    try:
        raw    = adapter.fetch(sid, '2018-01-01', END)
        normed = adapter.normalize(raw)
        vel    = compute_velocity(normed, window=5)

        ax2 = ax.twinx()
        ax.plot(raw.index, raw.values, color='steelblue', lw=1.5, label=label)
        ax2.plot(vel.index, vel.values, color='darkorange', lw=1.2, alpha=0.9,
                 label='Velocity')
        ax2.axhline(0, color='black', lw=0.5, ls='--')
        ax2.fill_between(vel.index, vel.values, 0,
                         where=(vel.values > 0), alpha=0.15, color='red',
                         label='Rising')
        ax2.fill_between(vel.index, vel.values, 0,
                         where=(vel.values < 0), alpha=0.15, color='green',
                         label='Falling')
        ax.set_ylabel(label, fontsize=9)
        ax2.set_ylabel('Velocity', fontsize=9, color='darkorange')
        ax.set_title(label, fontsize=10, fontweight='bold')
    except Exception as e:
        ax.set_title(f'{sid}: {e}')

plt.suptitle('Inflation Indicators — Spectral Velocity\n'
             '(Red fill = rising inflation velocity, Green = falling)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/inflation_velocity.png', dpi=150)
plt.show()

## Macro Conviction Heatmap by Category

In [ ]:
categories = {
    'Inflation':   ['CPIAUCSL','CPILFESL','PCEPILFE','PCEPI'],
    'Employment':  ['UNRATE','PAYEMS','ICSA','MANEMP'],
    'Growth':      ['GDPC1','INDPRO','TCU','DGORDER'],
    'Consumer':    ['UMCSENT','RSAFS'],
    'Housing':     ['HOUST','PERMIT'],
}

rows = []
for cat, sids in categories.items():
    for sid in sids:
        match = df[df['series_id'] == sid]
        if not match.empty:
            rows.append({
                'category': cat,
                'label':    match.iloc[0]['label'],
                'conviction': match.iloc[0]['conviction'],
            })

if rows:
    heat_df  = pd.DataFrame(rows)
    pivot    = heat_df.pivot_table(index='category', columns='label',
                                   values='conviction', aggfunc='first')

    fig, ax = plt.subplots(figsize=(max(10, len(pivot.columns)*1.2), 5))
    cmap = sns.diverging_palette(10, 133, as_cmap=True)
    sns.heatmap(pivot, annot=True, fmt='+.0f', cmap=cmap, center=0,
                vmin=-4, vmax=4, linewidths=0.5, ax=ax,
                cbar_kws={'label': 'Conviction Score'})
    ax.set_title('Macro Velocity Conviction by Category', fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    plt.tight_layout()
    plt.savefig('../results/macro_conviction_heatmap.png', dpi=150)
    plt.show()

In [ ]:
save_leaderboard(df, '../results/macro_leaderboard.csv')
print('Saved.')